In [17]:
from IPython.display import Image, display, HTML
import os
import base64
# 导入读取.env文件的库
from dotenv import load_dotenv
from types import SimpleNamespace
import requests
from tqdm.notebook import tqdm

In [ ]:
# 这段代码强制移除输出区域的高度限制，让所有内容直接铺开
display(HTML("""
<style>
    /* 针对代码输出块 */
    div.output_area pre {
        max-height: none !important;  /* 移除最大高度限制 */
        overflow: visible !important; /* 允许内容溢出，不出现滚动条 */
        white-space: pre-wrap;        /* 允许自动换行（如果你希望长行换行）*/
    }
    /* 针对文本输出块 */
    div.output_text {
        max-height: none !important;
        overflow: visible !important;
    }
</style>
"""))
# 禁用输出自动折叠
# 'last' (默认值)：只打印单元格中最后一个表达式的返回值。这是正常模式。
# 'all' ：打印单元格中每一个独立语句/表达式的返回值。
# 'none'：不自动打印任何返回值。
%config InteractiveShell.ast_node_interactivity='last'

# DEBUG	10	调试信息	最详细。包含程序运行的每一步细节、变量状态等。仅在开发调试时使用，生产环境通常关闭。
# INFO	20	一般信息    确认程序按预期运行。例如：“任务开始”、“文件加载成功”、“下载了 1MB”。适合日常监控。
# WARNING	30	警告 (默认级别)	表示发生了意外，但程序还能继续运行。例如：“磁盘空间不足”、“使用了废弃的函数”。这是 Jupyter 的默认设置。
# ERROR	40	错误	程序因某些问题无法执行某项功能。例如：“文件未找到”、“网络连接失败”。
# CRITICAL	50	严重错误	极严重的错误，可能导致程序中止。例如：“数据库崩溃”、“内存溢出”。
# %config Application.log_level="INFO"

In [3]:
def download_image(image_url, save_dir, filename=None):
    """
    从URL下载图片到指定目录（修复文件名含特殊字符问题）
    :param image_url: 图片URL
    :param save_dir: 保存目录（不存在则自动创建）
    :param filename: 自定义文件名（默认用时间戳+合法后缀）
    :return: 保存的文件路径
    """
    # 1. 创建保存目录（不存在则创建）
    os.makedirs(save_dir, exist_ok=True)
    
    # 2. 生成合法的默认文件名（核心修复：过滤URL参数+特殊字符）
    if not filename:
        import time
        timestamp = str(int(time.time()))
        
        # 第一步：截断URL中?后的参数部分（只保留纯图片路径）
        clean_url = image_url.split("?")[0]  # 去掉?及后面的所有内容
        # 第二步：提取图片后缀（只保留最后一个.后的合法后缀）
        file_ext = clean_url.split(".")[-1]
        # 第三步：过滤后缀中的非法字符（只保留字母/数字）
        file_ext = ''.join([c for c in file_ext if c.isalnum()])
        # 兜底：如果后缀为空，默认用png
        if not file_ext:
            file_ext = "png"
        
        # 最终文件名：时间戳+合法后缀
        filename = f"image_{timestamp}.{file_ext}"
    
    # 3. 拼接完整保存路径（确保路径合法）
    save_path = os.path.join(save_dir, filename)
    # 额外：替换路径中的非法字符（兜底）
    save_path = save_path.replace("?", "").replace("&", "").replace("=", "").replace("%", "")
    
    try:
        # 4. 发送请求下载（添加超时+用户代理，避免被拦截）
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }
        response = requests.get(image_url, headers=headers, timeout=30, stream=True)
        response.raise_for_status()  # 捕获HTTP错误（如404/500）
        
        # 5. 写入文件（分块写入，支持大文件）
        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print(f"✅ 图片下载完成！保存路径：{save_path}")
        return save_path
    
    except requests.exceptions.RequestException as e:
        print(f"❌ 下载失败：{str(e)}")
        return None
    except Exception as e:
        print(f"❌ 保存文件失败：{str(e)}")
        return None

def image_to_base64(image_path):
    """本地图片转Base64编码"""
    try:
        with open(image_path, "rb") as image_file:
            base64_str = base64.b64encode(image_file.read()).decode("utf-8")
        return base64_str
    except FileNotFoundError:
        print(f"错误：未找到图片文件 {image_path}")
        return None
    except Exception as e:
        print(f"图片转换失败：{str(e)}")
        return None

In [4]:
API_KEY = None
BASE_URL = None
load_dotenv(override=True)

True

In [5]:
args = SimpleNamespace(
    MODE_SOURCE = "WHATAI",
    DOWNLOAD_DIR = "./generated_images",
    CHARACTERS_IMAGES = "./characters_images",
    )

In [ ]:
if args.MODE_SOURCE == "WHATAI":
    # 读取环境变量中的WHATAI_API_KEY
    API_KEY = os.environ.get("WHATAI_API_KEY")

    # 校验API Key是否有效
    if not API_KEY:
        raise ValueError("未在.env文件中找到WHATAI_API_KEY，请检查配置")

    BASE_URL = os.environ.get("WHATAI_BASE_URL")

    # 校验API Key是否有效
    if not BASE_URL:
        raise ValueError("未在.env文件中找到WHATAI_BASE_URL，请检查配置")

    local_image_path = "./generated_images/image_test.png"  # 示例：D:/images/test.png
    image_base64 = image_to_base64(local_image_path)

    payload = {
        "prompt": "The girl is running on the street, and the background is a cityscape. The style is cyberpunk.",
        "model": "sora-2",
        "images": [
           image_base64,
        ],
        "aspect_ratio": "16:9",
        "hd": False,
        "duration": "10",
        # "notify_hook": "string",
        # "watermark": True,
        # "private": True
        }

    headers = {
    'Authorization': 'Bearer ' + API_KEY,
    'Content-Type': 'application/json'
    }

    response = requests.post(BASE_URL+"/v2/videos/generations", json=payload, headers=headers)

    print(response.status_code)
    print(response.text)

In [ ]:
print(response.json())
print(type(response.json()))
task_id = response.json()["task_id"]


In [8]:
headers = {
    'Authorization': 'Bearer ' + API_KEY,
    'Content-Type': 'application/json' # 通常建议加上内容类型
}

# 3. 发送 GET 请求
# payload 在原代码中为空字符串，GET 请求通常不需要 body，所以这里直接省略 data 参数
try:
    response = requests.get(BASE_URL+"/v2/videos/generations/"+task_id, headers=headers)
    
    # 4. 检查响应状态码
    if response.status_code == 200:
        # 尝试解析为 JSON (API 通常返回 JSON)
        try:
            data = response.json()
            print("成功获取数据 (JSON格式):")
            print(data)
        except ValueError:
            # 如果不是 JSON，则打印原始文本
            print("成功获取数据 (文本格式):")
            print(response.text)
    else:
        print(f"请求失败，状态码: {response.status_code}")
        print(f"错误信息: {response.text}")

except Exception as e:
    print(f"发生异常: {e}")

成功获取数据 (JSON格式):
{'task_id': 'video_H5XTPb6YPUkkchxwnbivGOQHjwrF1lNz', 'platform': 'openai', 'action': 'sora-video', 'status': 'SUCCESS', 'fail_reason': '', 'submit_time': 1773279575, 'start_time': 1773279586, 'finish_time': 1773279818, 'progress': '100%', 'data': {'output': 'https://oss.filenest.top/uploads/5f781fe4-1470-4d1f-a4e4-8676e2551068.mp4'}, 'cost': 1.25, 'search_item': ''}


In [19]:
# 1. 提取视频链接
# 注意路径：data -> data (键) -> output (键)
try:
    video_url = data['data']['output']
    print(f"发现视频链接: {video_url}")
except KeyError:
    print("错误：数据结构中没有找到 'data.output' 字段")
    exit()

# 2. 定义保存路径
# 可以固定文件名，也可以从 URL 中提取文件名
filename = "downloaded_video.mp4" 
# 或者动态提取文件名: filename = os.path.basename(video_url.split('?')[0])

print(f"开始下载保存到: {filename} ...")

# 3. 发起下载请求 (关键：使用 stream=True)
try:
    # 加上 headers 防止某些服务器拒绝非浏览器请求 (可选，但推荐)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    response = requests.get(video_url, stream=True, headers=headers)
    
    # 检查链接是否有效
    if response.status_code == 200:
        # 获取文件总大小 (如果服务器提供了 Content-Length)
        total_size = int(response.headers.get('content-length', 0))
        
        with open(filename, 'wb') as f, tqdm(
            desc=filename,               # 进度条左侧的描述文字
            total=total_size,            # 总字节数
            unit='B',                    # 单位：Bytes
            unit_scale=True,             # 自动转换为 KB, MB, GB
            unit_divisor=1024,           # 换算率
            ncols=300,                     # 进度条宽度
        ) as bar:
            # chunk_size=1048576  表示每次读取 1MB，避免一次性读入内存
            for chunk in response.iter_content(chunk_size=1048576):
                if chunk: # 过滤掉保持连接的空块
                    f.write(chunk)
                    bar.update(len(chunk))
        
        print(f"\n✅ 下载完成！文件已保存为: {filename}")
        
    else:
        print(f"❌ 下载失败，状态码: {response.status_code}")
        
except Exception as e:
    print(f"❌ 发生异常: {e}")

发现视频链接: https://oss.filenest.top/uploads/5f781fe4-1470-4d1f-a4e4-8676e2551068.mp4
开始下载保存到: downloaded_video.mp4 ...


downloaded_video.mp4:   0%|                                                                                   …

1048576

True

1048576

...

True

1048576

True

302061


✅ 下载完成！文件已保存为: downloaded_video.mp4


In [9]:

if args.MODE_SOURCE == "ARK":
    from volcenginesdkarkruntime import Ark
    # 读取环境变量中的ARK_API_KEY
    API_KEY = os.environ.get("ARK_API_KEY")

    # 校验API Key是否有效
    if not API_KEY:
        raise ValueError("未在.env文件中找到ARK_API_KEY，请检查配置")

    BASE_URL = os.environ.get("ARK_BASE_URL")

    # 校验API Key是否有效
    if not BASE_URL:
        raise ValueError("未在.env文件中找到ARK_BASE_URL，请检查配置")
    
    client = Ark(
        # The base URL for model invocation
        base_url=BASE_URL, 
        # Get API Key：https://console.volcengine.com/ark/region:ark+cn-beijing/apikey
        api_key=API_KEY, 
    )
    
    imagesResponse = client.images.generate( 
        # Replace with Model ID
        model="doubao-seedream-5-0-260128",
        prompt="充满活力的特写编辑肖像，模特眼神犀利，头戴雕塑感帽子，色彩拼接丰富，眼部焦点锐利，景深较浅，具有Vogue杂志封面的美学风格，采用中画幅拍摄，工作室灯光效果强烈。",
        size="2K",
        output_format="png",
        response_format="url",
        watermark=False
    )
    print(imagesResponse.data[0].url)
    image_url = imagesResponse.data[0].url
    display(Image(url=image_url, width=400))
    download_image(image_url, args.DOWNLOAD_DIR)

    # try:
    #     # 2. 处理本地图片（替换成你的图片路径）
    #     local_image_path = "./generated_images/image_test.png"  # 示例：D:/images/test.png
    #     image_base64 = image_to_base64(local_image_path)
    #     if not image_base64:
    #         exit(1)

    #     # 3. 调用Seedance接口
    #     print("----- 开始创建视频任务 -----")
    #     resp = client.content_generation.tasks.create(
    #         model="doubao-seedance-1-5-pro-251215",  # 你的模型ID
    #         content=[
    #             {
    #                 "text": (
    #                     "女孩抱着狐狸，女孩睁开眼，温柔地看向镜头，狐狸友善地抱着，镜头缓缓拉出，女孩的头发被风吹动"
    #                 ),
    #                 "type": "text"
    #             },
    #             {
    #                 "image_url": image_url,  # 用URL替代base64
    #                 "type": "image_url"      # 类型改为image_url
    #             }
    #         ],
    #         generate_audio=True,
    #         ratio="adaptive",
    #         duration=5,
    #         watermark=False,
    #     )
        
    #     print("任务创建成功！响应结果：")
    #     print(resp)

    # except Exception as e:
    #     print(f"程序执行失败：{str(e)}")
    #     exit(1)
